# Passo a passo do dashboard de sono
Este notebook é uma demonstração guiada do dashboard eywa_trees sobre o conjunto Sleep Health and Lifestyle. Carregamos os dados uma vez e depois mostramos como explorar a ferramenta tanto para uma tarefa de regressão quanto para uma de classificação.


In [ ]:
import threading

import pandas as pd

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split

from eywa_trees import EywaTreesDash

seed = 7


def start_dashboard(app, port: int):
    thread = threading.Thread(target=lambda: app.run(port=port, debug=False), daemon=True)
    thread.start()
    print(f"Dashboard available at http://127.0.0.1:{port}")
    return thread


## Dataset em um minuto
- Sleep Health and Lifestyle (374 pessoas) do Kaggle; coloque o CSV em `data/Sleep_health_and_lifestyle_dataset.csv`.
- Alvos usados aqui: `sleep-duration` (horas, regressão) e `sleep-disorder` (None / Insomnia / Sleep Apnea, classificação).
- Atributos mantidos: dados demográficos, ocupação, níveis de atividade e estresse, categoria de IMC, frequência cardíaca, passos diários e pressão arterial dividida em sistólica/diastólica. Só aplicamos limpeza leve e one-hot encoding.


In [ ]:
def clean_feature_names(df: pd.DataFrame) -> pd.DataFrame:
    cols = df.columns.str.lower()
    cols = cols.str.replace(" ", "-", regex=False)
    cols = cols.str.replace("_", "-", regex=False)
    cols = cols.str.replace("--", "-", regex=False)
    df.columns = cols
    return df


def load_sleep(path: str = "data/Sleep_health_and_lifestyle_dataset.csv") -> pd.DataFrame:
    sleep = pd.read_csv(path, index_col="Person ID")
    sleep["Sleep Disorder"] = sleep["Sleep Disorder"].fillna("None").astype("category")
    sleep["Gender"] = sleep["Gender"].astype("category")
    sleep["Occupation"] = sleep["Occupation"].astype("category")
    sleep["BMI Category"] = sleep["BMI Category"].astype("category")
    sleep["Sleep Duration"] = sleep["Sleep Duration"].astype(float)
    sleep["Quality of Sleep"] = sleep["Quality of Sleep"].astype(int)
    sleep["Physical Activity Level"] = sleep["Physical Activity Level"].astype(int)
    sleep["Stress Level"] = sleep["Stress Level"].astype(int)
    sleep["Heart Rate"] = sleep["Heart Rate"].astype(int)
    sleep["Daily Steps"] = sleep["Daily Steps"].astype(int)
    sleep[["systolic", "diastolic"]] = sleep["Blood Pressure"].str.split("/", expand=True).astype(int)
    sleep = sleep.drop(columns="Blood Pressure")
    return clean_feature_names(sleep)


def encode_features(df: pd.DataFrame, target: str) -> pd.DataFrame:
    X = df.drop(columns=[target])
    cat_cols = X.select_dtypes(include=["category"]).columns
    X = pd.get_dummies(X, columns=cat_cols, prefix_sep="_is_", dtype=int)
    X.columns = (
        X.columns.str.lower()
        .str.replace(" ", "-", regex=False)
        .str.replace("_", "-", regex=False)
        .str.replace("--", "-", regex=False)
    )
    return X


In [ ]:
sleep = load_sleep()
sleep.head()


## Regressão: prever horas noturnas
Usamos `sleep-duration` como alvo, retiramos a pontuação de survey `quality-of-sleep`, aplicamos one-hot encoding nos categóricos restantes e treinamos uma pequena random forest. O dashboard roda em thread de fundo para você continuar rolando.


In [ ]:
reg_target = "sleep-duration"

reg_df = sleep.drop(columns=["quality-of-sleep"])
reg_X = encode_features(reg_df, reg_target)
reg_y = reg_df[reg_target]

reg_X_train, reg_X_test, reg_y_train, reg_y_test = train_test_split(
    reg_X, reg_y, test_size=0.25, random_state=seed
)

reg_model = RandomForestRegressor(
    n_estimators=60,
    max_depth=6,
    min_samples_leaf=4,
    random_state=seed,
)
reg_model.fit(reg_X_train, reg_y_train)

reg_dash = EywaTreesDash(
    reg_model,
    reg_X_train,
    reg_X_test,
    reg_y_test,
    feature_names=reg_X.columns,
)
reg_dash.config(show_text=True)

reg_thread = start_dashboard(reg_dash, port=8061)


### Aba Model — visão Sankey de uma árvore

A aba Model expõe a geometria da floresta, uma árvore por vez. O controle deslizante superior permite escolher qual árvore focar a cada momento. O controle à esquerda aplica uma poda de profundidade no nível da visualização, atualizando automaticamente todos os valores.

Os nós são representados por elementos retangulares, enquanto as arestas representam relações de pai e filho. O layout vai da esquerda → direita (raiz para folhas). Cada divisão emite dois fluxos: por convenção, divisões para baixo representam amostras com feature ≤ threshold, e para cima representam amostras com feature > threshold. Passe o mouse sobre um nó para ver a feature e o limiar. Essa convenção reflete uma rotação de 90º no sentido anti-horário da convenção de layout do scikit-learn (esquerda como feature ≤ threshold e direita como feature > threshold).

As larguras das arestas mostram quantas linhas de treino passam por aquele ramo (`n_train`); folhas com zero linhas de treino ficam ocultas para que divisões finas e não usadas sumam.

As cores codificam as horas de sono previstas naquele nó; nós anteriores mostram a média ponderada dos filhos após a poda.

A métrica no canto superior esquerdo mostra o desempenho que a árvore atual (com o limite de profundidade aplicado) teria no conjunto de teste.


### Aba Predict — explore valores de atributos interativamente

A aba Predict permite fabricar registros individuais e inspecionar seu caminho pela floresta.

A coluna da esquerda expõe diversos controles deslizantes representando as features do conjunto de dados. Uma amostragem por quantis garante que os marcadores de ticks respeitem os valores possíveis, mantendo espaçamentos regulares. O usuário pode sintetizar sua própria entrada ajustando esses controles. "Return to median" redefine todos os sliders para o índice mediano.

À direita, um Go plot padrão desenha uma árvore por vez com um seletor de árvore; as marcações são afinadas para florestas grandes, mas todo id de árvore permanece selecionável.

O texto de previsão sobreposto vem do random forest completo aplicado à linha sintetizada a partir das configurações dos sliders.

As árvores são deslocadas de cima → para baixo, respeitando a convenção do sklearn (fluxo à esquerda respeita `feature ≤ threshold`, enquanto fluxo à direita respeita `feature > threshold`).

Após sintetizar um elemento de entrada, seu traçado dentro da árvore real é destacado.


### Aba Boundary — paisagem latente

A aba Boundary relaciona uma entrada sintética com sua posição na fronteira de predição do modelo.

Os sliders à esquerda são os mesmos da aba Predict.

O mapa à direita é uma projeção latente 2D do treino (t-SNE), colorida pelas previsões do modelo.

Clique em qualquer ponto do mapa: a cruz fica exatamente onde você clicou, o ponto de treino mais próximo é decodificado para os bins de features, e sliders/previsão são ajustados para ele. Movendo os sliders também reposiciona a cruz para o perfil de treino mais próximo no espaço latente.


### Aba Paths — cobertura das principais regras de decisão

A aba Paths expõe estatísticas sobre os caminhos mais comuns — a sequência de regras que liga o nó raiz a uma folha.

Um caminho da floresta original representa uma única folha em uma única árvore da floresta. Eles são agrupados em caminhos representativos pela cobertura de treinamento e depois ranqueados pelo número de amostras que cobrem.

Para cada Pathway exibido, `n_trees` representa o número de caminhos originais que se agrupam para formar o exibido. `branch_prob_forest` mostra uma cobertura média: entre todos os caminhos agrupados, a proporção de amostras que alcançam aquela folha.

Assim, uma linha única nesta aba representa não só um caminho de uma árvore, mas sim um padrão que existe em várias árvores da floresta. Os caminhos no topo são mais representativos que os de baixo.

Cada caminho mostra as regras aplicadas e a previsão à direita.


## Classificação: sinalizando distúrbios do sono
Trocamos o alvo para `sleep-disorder` (None / Insomnia / Sleep Apnea), mantemos o mesmo pré-processamento e treinamos um random forest para mostrar como o dashboard se comporta para classificação.


In [ ]:
class_target = "sleep-disorder"

class_df = sleep.copy()
class_X = encode_features(class_df, class_target)
class_y = class_df[class_target]
class_names = sorted(class_y.unique())

class_X_train, class_X_test, class_y_train, class_y_test = train_test_split(
    class_X, class_y, test_size=0.25, random_state=seed, stratify=class_y
)

class_model = RandomForestClassifier(
    n_estimators=80,
    max_depth=6,
    min_samples_leaf=4,
    random_state=seed,
)
class_model.fit(class_X_train, class_y_train)

class_dash = EywaTreesDash(
    class_model,
    class_X_train,
    class_X_test,
    class_y_test,
    feature_names=class_X.columns,
    class_names=class_names,
)
class_dash.config(show_text=True)

class_thread = start_dashboard(class_dash, port=8062)


### Como ler o dashboard de classificação
- **Model tab**: a coloração do Sankey passa a ser a classe prevista; a métrica vira acurácia no conjunto de validação. Os sliders de profundidade e árvore funcionam como antes, permitindo ver quais árvores estão mais confiantes sobre um distúrbio.
- **Predict tab**: a sobreposição mostra o rótulo de classe previsto; as cores no Go plot seguem a classe majoritária em cada nó. Use para investigar trocas de classe ao alternar variáveis de estilo de vida.
- **Rules tab**: lista clusters de regras com cobertura e predição; selecione uma linha para destacar o cluster na árvore.
- As probabilidades de classe nos nós internos agregam as probabilidades das folhas ponderadas pelas contagens de treino, então a poda deixa claro quais divisões iniciais já separam bem os distúrbios.
